In [1]:
import pandas as pd
import numpy as np

data_url = "http://lib.stat.cmu.edu/datasets/boston"
raw_df = pd.read_csv(data_url, sep="\s+", skiprows=22, header=None)

# The data is interleaved with features on one line and the target on the next.
# We need to separate them.
data = raw_df.values[::2, :]
target = raw_df.values[1::2, 2] # The target is in the 3rd column (index 2) of the second row

# The last two columns of the data are combined in the raw data, so we need to split them
data = np.hstack([data[:, :9], data[:, 9:11]])

# Convert data and target to float, coercing errors
data = data.astype(float)
target = target.astype(float)

<>:5: SyntaxWarning: invalid escape sequence '\s'
<>:5: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipython-input-1249776968.py:5: SyntaxWarning: invalid escape sequence '\s'
  raw_df = pd.read_csv(data_url, sep="\s+", skiprows=22, header=None)


In [2]:
# Check for and remove rows with NaN in y
nan_mask = np.isnan(target)
X = data[~nan_mask]
y = target[~nan_mask]

In [3]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score

In [4]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [5]:
X

array([[6.3200e-03, 1.8000e+01, 2.3100e+00, ..., 1.0000e+00, 2.9600e+02,
        1.5300e+01],
       [2.7310e-02, 0.0000e+00, 7.0700e+00, ..., 2.0000e+00, 2.4200e+02,
        1.7800e+01],
       [2.7290e-02, 0.0000e+00, 7.0700e+00, ..., 2.0000e+00, 2.4200e+02,
        1.7800e+01],
       ...,
       [6.0760e-02, 0.0000e+00, 1.1930e+01, ..., 1.0000e+00, 2.7300e+02,
        2.1000e+01],
       [1.0959e-01, 0.0000e+00, 1.1930e+01, ..., 1.0000e+00, 2.7300e+02,
        2.1000e+01],
       [4.7410e-02, 0.0000e+00, 1.1930e+01, ..., 1.0000e+00, 2.7300e+02,
        2.1000e+01]])

In [6]:
y

array([24. , 21.6, 34.7, 33.4, 36.2, 28.7, 22.9, 27.1, 16.5, 18.9, 15. ,
       18.9, 21.7, 20.4, 18.2, 19.9, 23.1, 17.5, 20.2, 18.2, 13.6, 19.6,
       15.2, 14.5, 15.6, 13.9, 16.6, 14.8, 18.4, 21. , 12.7, 14.5, 13.2,
       13.1, 13.5, 18.9, 20. , 21. , 24.7, 30.8, 34.9, 26.6, 25.3, 24.7,
       21.2, 19.3, 20. , 16.6, 14.4, 19.4, 19.7, 20.5, 25. , 23.4, 18.9,
       35.4, 24.7, 31.6, 23.3, 19.6, 18.7, 16. , 22.2, 25. , 33. , 23.5,
       19.4, 22. , 17.4, 20.9, 24.2, 21.7, 22.8, 23.4, 24.1, 21.4, 20. ,
       20.8, 21.2, 20.3, 28. , 23.9, 24.8, 22.9, 23.9, 26.6, 22.5, 22.2,
       23.6, 28.7, 22.6, 22. , 22.9, 25. , 20.6, 28.4, 21.4, 38.7, 43.8,
       33.2, 27.5, 26.5, 18.6, 19.3, 20.1, 19.5, 19.5, 20.4, 19.8, 19.4,
       21.7, 22.8, 18.8, 18.7, 18.5, 18.3, 21.2, 19.2, 20.4, 19.3, 22. ,
       20.3, 20.5, 17.3, 18.8, 21.4, 15.7, 16.2, 18. , 14.3, 19.2, 19.6,
       23. , 18.4, 15.6, 18.1, 17.4, 17.1, 13.3, 17.8, 14. , 14.4, 13.4,
       15.6, 11.8, 13.8, 15.6, 14.6, 17.8, 15.4, 21

In [8]:
X.shape

(506, 11)

In [9]:
y.shape

(506,)

In [10]:
lr = LinearRegression()
dt = DecisionTreeRegressor()
knn = KNeighborsRegressor()

In [11]:
lr.fit(X_train, y_train)
dt.fit(X_train, y_train)
knn.fit(X_train, y_train)

KNeighborsRegressor()

In [12]:
y_pred_lr = lr.predict(X_test)
y_pred_dt = dt.predict(X_test)
y_pred_knn = knn.predict(X_test)

In [14]:
print(r2_score(y_test, y_pred_lr))
print(r2_score(y_test, y_pred_dt))
print(r2_score(y_test, y_pred_knn))

0.6041513582037034
0.8598150310182118
0.460738762376261


In [15]:
from sklearn.ensemble import BaggingRegressor

beg__regressor = BaggingRegressor(random_state=1)
bag_regressor = beg__regressor.fit(X_train, y_train)
y_pred_bag = bag_regressor.predict(X_test)

In [18]:
print(bag_regressor.score(X_train, y_train))
print(bag_regressor.score(X_test, y_test))

0.9468367581054066
0.7684555780485147


In [22]:
params = {'estimator':[None,LinearRegression(), DecisionTreeRegressor()],
          'n_estimators':[20,50,100],
          'max_samples':[0.5,1],
          'max_features':[0.5,1],
          'bootstrap':[True,False],
          'bootstrap_features':[True,False]
}
bagging_regressor = GridSearchCV(BaggingRegressor(random_state=1), param_grid=params, cv=5)
bagging_regressor.fit(X_train, y_train)

print(bagging_regressor.best_params_)
print(bagging_regressor.best_score_)

{'bootstrap': False, 'bootstrap_features': False, 'estimator': None, 'max_features': 0.5, 'max_samples': 0.5, 'n_estimators': 100}
0.7175739391393321
